<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/SK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

GPU available: True
Thu Oct 15 14:01:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [2]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q "qwen-omni-utils[decord]" pymupdf python-docx pandas
# Optional (only if you set SHOW_EMBED = True below):
# !pip install -q sentence-transformers scikit-learn
# If the Qwen2.5-Omni import fails, use the dev build instead:
# !pip install -q git+https://github.com/huggingface/transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 95.2 MB/s eta 0:00:00


In [3]:
import os, gc, glob, re, time, hashlib, subprocess, math
from concurrent.futures import ThreadPoolExecutor
import torch, pymupdf, docx
import numpy as np, pandas as pd
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor, BitsAndBytesConfig
from qwen_omni_utils import process_mm_info

assert torch.cuda.is_available(), "No GPU: set the runtime type to T4 GPU and restart."

model_id = "Qwen/Qwen2.5-Omni-3B"
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"      # required for batched generation and last-token logits

def load(quantize_4bit=False):
    kwargs = dict(torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
    if quantize_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4",
            llm_int8_skip_modules=["audio_tower", "visual", "lm_head"])
    return Qwen2_5OmniForConditionalGeneration.from_pretrained(model_id, **kwargs)

try:
    model = load(quantize_4bit=False); print("Loaded fp16")
except torch.cuda.OutOfMemoryError:
    gc.collect(); torch.cuda.empty_cache()
    model = load(quantize_4bit=True); print("fp16 didn't fit, loaded 4-bit")

if hasattr(model, "disable_talker"):      # text output only: free the speech modules
    model.disable_talker()
thinker = model.thinker
torch.cuda.empty_cache()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

chat_template.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

tokenizer_config.json:   0%|          | 0.00/6.47k [00:00<?, ?B/s]

...

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

spk_dict.pt: reconstructing file:   0%|          |  0.00B /  260kB            

spk_dict.pt: downloading bytes:           |  0.00B            

Loaded fp16
GPU memory used: 9.4 GB


In [4]:
IMG = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
VID = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
AUD = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
TXT = {".txt", ".md", ".csv", ".json", ".html"}

# ---- text / PDF ----
MAX_CHARS, MAX_PIECES = 12000, 8       # long text: summarize up to MAX_PIECES pieces spread across the document

# ---- video ----
SHORT_VIDEO_SEC = 30      # videos up to this length are sent to the model whole
CLIP_SEC        = 10      # length of each sampled clip for longer videos
MAX_CLIPS       = 6       # at most this many clips per video (speed vs. coverage knob)
CLIP_HEIGHT     = 360     # clips are downscaled to this height before decoding

# ---- SPEED PRESET: "fast" | "balanced" | "quality" ----
# Visual tokens ~ (nframes / 2) * (max_pixels / 784). Prefill time grows with it.
#   quality  : 24 frames @ 360x420 -> ~2300 visual tokens  (V2 behaviour)
#   balanced : 12 frames @ 280x280 -> ~600
#   fast     :  8 frames @ 224x224 -> ~260
PRESET = "fast"
SUMMARY_SENTENCES = 1     # shorter generated summary = fewer decode steps (1 is fastest, 2-3 = more detail)
VIDEO_MODE = "single"     # "single": ONE model call per video, frames spread over the whole video (fastest)
                          # "clips" : describe up to MAX_CLIPS clips, then summarize the notes (slower, more temporal detail)
MAX_FRAME_MULT = 3        # in "single" mode, longer videos get up to this many times the preset's frame count
PRESCALE_SHORT = False    # True: downscale + 4 fps via ffmpeg before decoding (helps if profile shows slow decode_media)

# Tiers from best to cheapest. On OOM the code steps down this list; STATE remembers what worked.
# min_pixels is set explicitly: newer qwen-omni-utils defaults it to ~100k pixels, which is larger
# than the max_pixels of the cheaper tiers and triggers "max_pixels must be >= min_pixels".
MIN_PX = 16 * 28 * 28
TIERS = [
    {"nframes": 24, "max_pixels": 360 * 420, "min_pixels": MIN_PX},   # quality
    {"nframes": 12, "max_pixels": 280 * 280, "min_pixels": MIN_PX},   # balanced
    {"nframes": 8,  "max_pixels": 224 * 224, "min_pixels": MIN_PX},   # fast
    {"nframes": 8,  "max_pixels": 168 * 168, "min_pixels": MIN_PX},   # emergency
]
_start = {"quality": 0, "balanced": 1, "fast": 2}[PRESET]
STATE = {"single_tier": _start, "clip_tier": max(_start, 1), "batch": 3}
FINAL_TOKENS = 30 + 30 * SUMMARY_SENTENCES   # output budget for TOPIC + SUMMARY

# ---- judge ----
JUDGE_THRESHOLD = 0.5     # a skill is matched when judge >= this. Tune it on a few labeled examples.
JUDGE_BATCH     = 8       # skills scored per forward pass (lower it if you hit OOM with many skills)
SHOW_EMBED      = False   # True adds an "embed" similarity column (needs sentence-transformers)

LABEL_EXAMPLES = "'Programming' (not 'Tower of Hanoi puzzle'), 'Chess' (not 'two players moving pieces'), 'Cooking', 'Dogs', 'Finance', 'Movies'"

SYSTEM = {"role": "system", "content": [{"type": "text",
    "text": "You are a concise assistant. Answer directly in plain text. Never ask the user questions."}]}

# One generation returns both the label and the summary
FINAL_PROMPT = ("Reply in exactly this format and nothing else:\n"
    "TOPIC: <1 or 2 words, the broad subject field, not a description. Examples: " + LABEL_EXAMPLES + ">\n"
    f"SUMMARY: <what this is about, at most {SUMMARY_SENTENCES} short sentences>")
CLIP_PROMPT = "Describe what happens in this clip in 2 short sentences."

In [5]:
def _cleanup():
    gc.collect(); torch.cuda.empty_cache()

PROFILE = {}   # seconds per stage (plus token counts), reset at the start of every check()
def _add(k, v): PROFILE[k] = PROFILE.get(k, 0) + v

def _generate(convs, max_new_tokens):
    """Run a batch of conversations through the thinker. Returns one string per conversation."""
    t0 = time.time()
    texts = processor.apply_chat_template(convs, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(convs, use_audio_in_video=False)   # video decoding happens here
    t1 = time.time()
    inputs = processor(text=texts, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True, use_audio_in_video=False)
    inputs = inputs.to("cuda").to(torch.float16)
    torch.cuda.synchronize(); t2 = time.time()
    with torch.inference_mode():
        out = thinker.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    torch.cuda.synchronize(); t3 = time.time()
    n_in = inputs.input_ids.shape[1]
    _add("decode_media_s", t1 - t0); _add("preprocess_s", t2 - t1); _add("prefill+generate_s", t3 - t2)
    _add("input_tokens", n_in * len(convs)); _add("new_tokens", (out.shape[1] - n_in) * len(convs))
    return [t.strip() for t in processor.batch_decode(out[:, n_in:], skip_special_tokens=True)]

def generate_many(contents, max_new_tokens=200, tier_key="single_tier"):
    """
    contents : list of content-part lists (one per request). Processed in batches of STATE["batch"].
    On OOM: first halve the batch size, then drop to a cheaper video tier. Both are remembered.
    """
    results, i = [None] * len(contents), 0
    while i < len(contents):
        group = contents[i:i + STATE["batch"]]
        has_video = any(c["type"] == "video" for content in group for c in content)
        convs = []
        for content in group:
            content = [dict(c) for c in content]
            for c in content:
                if c["type"] == "video":
                    m = c.pop("frame_mult", 1)                       # longer video -> more frames (still even)
                    c.update(TIERS[STATE[tier_key]])
                    c["nframes"] = max(4, int(round(c["nframes"] * m / 2)) * 2)
            convs.append([SYSTEM, {"role": "user", "content": content}])
        try:
            results[i:i + len(group)] = _generate(convs, max_new_tokens)
            i += len(group)
        except torch.cuda.OutOfMemoryError:
            _cleanup()
            if STATE["batch"] > 1:
                STATE["batch"] = max(1, STATE["batch"] // 2)
                print(f"Out of memory -> batch size now {STATE['batch']}")
            elif has_video and STATE[tier_key] < len(TIERS) - 1:
                STATE[tier_key] += 1
                print(f"Out of memory -> video tier now {TIERS[STATE[tier_key]]}")
            else:
                results[i] = "(out of memory)"; i += 1
    _cleanup()
    return results

def run_text(prompt, max_new_tokens=200):
    return generate_many([[{"type": "text", "text": prompt}]], max_new_tokens)[0]

# Warm-up: the first generate call pays one-time CUDA/kernel setup. Do it now so timings below are honest.
_t = time.time(); run_text("Say OK.", 4); PROFILE.clear()
print(f"Warm-up done in {time.time() - _t:.1f}s")

Warm-up done in 3.4s


In [6]:
# ---- images, audio, PDF, docx, text ----
def _ext(src):
    """File extension for an existing path or a URL; '' for plain typed-in text."""
    if src.startswith("http") or os.path.exists(src):
        return os.path.splitext(src.split("?")[0])[1].lower()
    return ""

def _spread(items, k):
    """Pick up to k items evenly across the list (beginning, middle and end)."""
    if len(items) <= k: return items
    idx = np.unique(np.linspace(0, len(items) - 1, k).astype(int))
    return [items[i] for i in idx]

def _text_content(text):
    if len(text) > MAX_CHARS:             # long text: summarize pieces (batched), then the notes
        pieces = _spread([text[i:i + MAX_CHARS] for i in range(0, len(text), MAX_CHARS)], MAX_PIECES)
        notes = generate_many([[{"type": "text", "text": f"{p}\n\nSummarize this part in 3 sentences."}]
                               for p in pieces], 100)
        text = "\n".join(f"Part {i + 1}: {n}" for i, n in enumerate(notes))
    return [{"type": "text", "text": f"{text}\n\n{FINAL_PROMPT}"}]

def _pdf_content(path):
    with pymupdf.open(path) as doc:
        text = "\n".join(page.get_text() for page in doc)
        if len(text.strip()) > 200:
            return _text_content(text)
        content = []                       # scanned PDF: first 3 pages as images
        for i, page in enumerate(doc):
            if i >= 3: break
            png = f"/content/_page{i}.png"
            page.get_pixmap(dpi=100).save(png)
            content.append({"type": "image", "image": png})
    return content + [{"type": "text", "text": FINAL_PROMPT}]

def build_content(src):
    """Non-video input -> model-ready content parts."""
    ext = _ext(src)
    if not ext and not src.startswith("http"):
        return _text_content(src)          # plain text typed in directly
    if ext in IMG: return [{"type": "image", "image": src}, {"type": "text", "text": FINAL_PROMPT}]
    if ext in AUD: return [{"type": "audio", "audio": src}, {"type": "text", "text": FINAL_PROMPT}]
    if ext == ".pdf": return _pdf_content(src)
    if ext == ".docx": return _text_content("\n".join(p.text for p in docx.Document(src).paragraphs))
    if ext in TXT:
        with open(src, errors="ignore") as f: return _text_content(f.read())
    raise ValueError(f"Unsupported file type: {ext}")

In [7]:
# ---- video: sample a few clips, describe them in batches, then one final summary ----
def video_duration(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", path], capture_output=True, text=True)
    return float(r.stdout.strip())

def clip_starts(duration):
    """Start times of up to MAX_CLIPS windows of CLIP_SEC, spread evenly across the video."""
    if duration <= MAX_CLIPS * CLIP_SEC:                       # short enough: contiguous coverage
        return [k * CLIP_SEC for k in range(math.ceil(duration / CLIP_SEC))]
    return [round((duration - CLIP_SEC) * k / (MAX_CLIPS - 1), 2) for k in range(MAX_CLIPS)]

def extract_clips(path, duration, out_dir="/content/clips"):
    """Cut only the sampled windows, downscaled and re-encoded fast (so cuts are exact, decode is cheap)."""
    os.makedirs(out_dir, exist_ok=True)
    for f in glob.glob(f"{out_dir}/*.mp4"): os.remove(f)
    starts = clip_starts(duration)
    def cut(k_s):
        k, s = k_s
        out = f"{out_dir}/clip_{k:02d}.mp4"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(s), "-t", str(CLIP_SEC), "-i", path,
                        "-an", "-vf", f"scale=-2:{CLIP_HEIGHT}", "-c:v", "libx264", "-preset", "ultrafast",
                        "-crf", "30", out], check=True)
        return out, s
    with ThreadPoolExecutor(max_workers=4) as ex:
        return list(ex.map(cut, enumerate(starts)))

def prescale(path, out="/content/prescaled.mp4"):
    """Optional: shrink a short video (low height, 4 fps) so frame decoding is cheap."""
    t = time.time()
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-an",
                    "-vf", f"scale=-2:{CLIP_HEIGHT},fps=4", "-c:v", "libx264", "-preset", "ultrafast",
                    "-crf", "30", out], check=True)
    _add("prescale_s", time.time() - t)
    return out

def _mmss(sec): return f"{int(sec // 60)}:{int(sec % 60):02d}"

def summarize_video(src):
    """Returns raw 'TOPIC: ...\nSUMMARY: ...' text."""
    duration = video_duration(src)
    if VIDEO_MODE == "single" or duration <= SHORT_VIDEO_SEC:
        video = prescale(src) if PRESCALE_SHORT else src
        mult = min(MAX_FRAME_MULT, max(1.0, duration / SHORT_VIDEO_SEC))
        content = [{"type": "video", "video": video, "frame_mult": mult}, {"type": "text", "text": FINAL_PROMPT}]
        return generate_many([content], FINAL_TOKENS, "single_tier")[0]
    _t = time.time()
    clips = extract_clips(src, duration)
    _add("extract_clips_s", time.time() - _t)
    notes = generate_many([[{"type": "video", "video": p}, {"type": "text", "text": CLIP_PROMPT}]
                           for p, _ in clips], 70, "clip_tier")
    joined = "\n".join(f"[{_mmss(s)}] {n}" for (_, s), n in zip(clips, notes))
    prompt = ("These are notes on clips sampled evenly across one video, with their timestamps:\n\n"
              f"{joined}\n\n{FINAL_PROMPT}")
    return generate_many([[{"type": "text", "text": prompt}]], FINAL_TOKENS, "clip_tier")[0]

def summarize(src):
    """Any input -> raw 'TOPIC: ...\nSUMMARY: ...' text."""
    if _ext(src) in VID: return summarize_video(src)
    return generate_many([build_content(src)], FINAL_TOKENS)[0]

In [8]:
tok = processor.tokenizer
def _ids(words): return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})
YES_IDS, NO_IDS = _ids(["YES", "Yes", "yes"]), _ids(["NO", "No", "no"])

JUDGE_SYSTEM = {"role": "system", "content": [{"type": "text",
    "text": "You are a strict classifier. Reply with exactly one word: YES or NO."}]}

def skill_text(skill): return f"{skill['name']}: {skill['description']}"

def clean_label(raw, max_words=3):
    label = raw.strip().split("\n")[0].strip(" \"'`*.:;-<>")
    if label.lower().startswith("label"):
        label = label.split(":", 1)[-1].strip(" \"'`*.")
    return " ".join(label.split()[:max_words])

def parse_output(raw):
    t = re.search(r"TOPIC:\s*(.+)", raw, re.I)
    s = re.search(r"SUMMARY:\s*(.+)", raw, re.I | re.S)
    return (clean_label(t.group(1)) if t else ""), (s.group(1).strip() if s else raw.strip())

def make_label(summary):
    """Fallback only: used when the combined generation did not return a TOPIC line."""
    q = (f"Content summary:\n{summary}\n\nWhat general subject or field is this content about? "
         f"Answer with 1 or 2 words only: the broad category, not a description. Examples: {LABEL_EXAMPLES}. "
         "Output only the category.")
    return clean_label(run_text(q, 8))

def judge_all(label, summary, skills):
    """P(YES) per skill from next-token logits: one batched forward pass per JUDGE_BATCH skills."""
    probs = []
    for i in range(0, len(skills), JUDGE_BATCH):
        prompts = []
        for s in skills[i:i + JUDGE_BATCH]:
            q = (f"Content topic: {label}\nContent summary: {summary}\n\nSkill: {skill_text(s)}\n\n"
                 "Is the content clearly about this skill's subject area, so that a person with this skill "
                 "would find it relevant? Answer YES or NO.")
            conv = [JUDGE_SYSTEM, {"role": "user", "content": [{"type": "text", "text": q}]}]
            prompts.append(processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False))
        inputs = processor(text=prompts, return_tensors="pt", padding=True, use_audio_in_video=False)
        inputs = inputs.to("cuda").to(torch.float16)
        with torch.inference_mode():
            try:    out = thinker(**inputs, use_cache=False, logits_to_keep=1)
            except TypeError: out = thinker(**inputs, use_cache=False)
            logits = out.logits[:, -1].float()
        yes, no = torch.logsumexp(logits[:, YES_IDS], 1), torch.logsumexp(logits[:, NO_IDS], 1)
        probs += torch.sigmoid(yes - no).tolist()
        del out, logits; _cleanup()
    return probs

_embedder = None
def embed_scores(summary, skills):
    global _embedder
    from sentence_transformers import SentenceTransformer, util
    if _embedder is None:
        _embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    e_s = _embedder.encode(summary, convert_to_tensor=True, normalize_embeddings=True)
    e_k = _embedder.encode([skill_text(s) for s in skills], convert_to_tensor=True, normalize_embeddings=True)
    return util.cos_sim(e_s, e_k)[0].tolist()

CACHE = {}   # (label, summary) per input, so re-checking with other skills/thresholds is nearly free
def _key(src):
    if os.path.exists(src):
        st = os.stat(src); return ("file", os.path.abspath(src), st.st_size, int(st.st_mtime))
    return ("str", hashlib.md5(src.encode()).hexdigest())

def check(src, skills, threshold=None, use_cache=True):
    """
    src    : text, or a path/URL to an image, video, audio, pdf, docx or text file
    skills : list of {"name": ..., "description": ...}
    returns: dict with related, matches, label, summary, table (per-skill scores), timing (seconds)
    """
    threshold = JUDGE_THRESHOLD if threshold is None else threshold
    timing, key = {}, _key(src)
    PROFILE.clear()
    t = time.time()
    if use_cache and key in CACHE:
        label, summary = CACHE[key]; timing["summarize"] = 0.0
    else:
        label, summary = parse_output(summarize(src))
        if not label: label = make_label(summary)
        CACHE[key] = (label, summary); timing["summarize"] = round(time.time() - t, 1)

    t = time.time()
    rows = [{"skill": s["name"], "judge": round(p, 3)} for s, p in zip(skills, judge_all(label, summary, skills))]
    table = pd.DataFrame(rows)
    if SHOW_EMBED: table["embed"] = [round(x, 3) for x in embed_scores(summary, skills)]
    table = table.sort_values("judge", ascending=False).reset_index(drop=True)
    table["matched"] = table.judge >= threshold
    timing["judge"] = round(time.time() - t, 1)
    matches = table[table.matched].skill.tolist()
    return {"related": bool(matches), "matches": matches, "label": label,
            "summary": summary, "table": table, "timing": timing,
            "profile": {k: round(v, 2) for k, v in PROFILE.items()}}

def show(r, verbose=False):
    print(f"Topic   : {r['label']}")
    print(f"Related : {r['related']}   Matched: {r['matches']}")
    print(f"Time (s): {r['timing']}")
    if r.get("profile"): print(f"Profile : {r['profile']}")
    if verbose: print(f"Summary : {r['summary']}")
    return r["table"]

## Real data from the backend


In [15]:
# ===== REAL DATA: user + skills + posts from the backend =====
import requests, json, os
from urllib.parse import urlparse, urljoin

API_BASE  = "https://evolve-backend-4lux.onrender.com"
USER_ID   = 1
ENDPOINT  = "/api/skills/user/{user_id}/skills-posts"
ALLOW_MOCK_FALLBACK = False         # real API: show the real error instead of silently using fake data
REQUEST_TIMEOUT = 90                # Render free instances sleep; the first request can take ~1 minute
MEDIA_DIR = "/content/media"

# Login token: Colab sidebar -> key icon (Secrets) -> name EVOLVE_TOKEN, notebook access ON.
# No secret = hidden prompt. Never write the token inside the notebook.
API_TOKEN = None
try:
    from google.colab import userdata
    API_TOKEN = userdata.get("EVOLVE_TOKEN")
except Exception:
    pass
if not API_TOKEN:
    from getpass import getpass
    API_TOKEN = getpass("Paste API token (leave empty if none): ").strip() or None

# The API returns "description": null for every skill, and the judge compares posts against descriptions.
# Until the backend fills them in, these are used when the API has none (matched by skill name).
SKILL_DESCRIPTIONS = {
    "Python":          "Python programming language: writing, debugging and explaining Python code, scripts, libraries and tools",
    "AI/ML":           "artificial intelligence and machine learning: models, neural networks, training, data science and LLMs",
    "Web Development": "building websites and web apps: HTML, CSS, JavaScript, frameworks, frontend and backend APIs",
    "Cloud":           "cloud computing: AWS, Azure, Google Cloud, servers, deployment, Docker, DevOps and infrastructure",
}

# Shape of the real response (trimmed). Only used if you set ALLOW_MOCK_FALLBACK = True.
MOCK_RESPONSE = {
    "success": True, "userId": 1, "user": {"id": 1, "name": "a_mor"},
    "skills": [
        {"id": 1, "skillId": 1, "relationshipType": "LEARNING", "skill": {"id": 1, "name": "Python", "description": None, "isActive": True}},
        {"id": 2, "skillId": 3, "relationshipType": "LEARNING", "skill": {"id": 3, "name": "AI/ML", "description": None, "isActive": True}},
    ],
    "posts": [{"id": 1, "caption": "My first EVOLV post", "postType": "IMAGE", "status": "PUBLISHED", "deletedAt": None,
               "media": [{"id": 1, "mediaType": "IMAGE", "mediaUrl": "https://example.com/post_1.png", "position": 0}]}],
}

def _first(d, keys, default=""):
    for k in keys:
        v = d.get(k)
        if v not in (None, ""): return v
    return default

def _headers():
    h = {}
    if API_TOKEN: h["Authorization"] = API_TOKEN if " " in API_TOKEN else f"Bearer {API_TOKEN}"
    return h

def fetch_user_data(user_id=USER_ID, allow_mock=None):
    allow_mock = ALLOW_MOCK_FALLBACK if allow_mock is None else allow_mock
    url = API_BASE.rstrip("/") + ENDPOINT.format(user_id=user_id)
    try:
        r = requests.get(url, headers=_headers(), timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        data = r.json()
        print(f"Fetched from {url}")
    except Exception as e:
        status = getattr(getattr(e, "response", None), "status_code", None)
        if status in (401, 403): print(f"HTTP {status}: the API needs a valid login token (see API_TOKEN above).")
        if not allow_mock: raise
        print(f"Could not reach {url}\n   {type(e).__name__}: {e}\n   -> using MOCK_RESPONSE instead")
        data = MOCK_RESPONSE
    return data.get("data", data) if isinstance(data, dict) else data

def normalize(data):
    """API JSON -> (user, skills, posts).
    skills: [{id, name, description}]            (the API nests each skill under "skill")
    posts : [{id, text, media:[{id,url,type}]}]  (caption is the text; media is a list, in order)"""
    user = data.get("user") or {"id": data.get("userId", USER_ID)}
    skills = []
    for i, us in enumerate(data.get("skills", [])):
        s = us.get("skill") or us
        if s.get("isActive") is False: continue
        name = _first(s, ["name", "title", "skill_name"])
        skills.append({"id": s.get("id", us.get("skillId", i)), "name": name,
                       "description": _first(s, ["description", "desc", "details"]) or SKILL_DESCRIPTIONS.get(name, name)})
    posts = []
    for i, p in enumerate(data.get("posts", [])):
        if p.get("deletedAt"): continue
        text = "\n".join(x for x in [_first(p, ["title"]), _first(p, ["caption", "content", "text", "body", "description"])] if x)
        media = [{"id": m.get("id"), "url": _first(m, ["mediaUrl", "media_url", "url"], None), "type": m.get("mediaType")}
                 for m in sorted(p.get("media") or [], key=lambda m: m.get("position") or 0)]
        posts.append({"id": p.get("id", i), "text": text, "media": [m for m in media if m["url"]], "type": p.get("postType")})
    return user, skills, posts

def get_media(m, post_id):
    """Download one media item to a local file (the model code needs a local path for image/video)."""
    url = m["url"]
    if not url.startswith("http"): url = urljoin(API_BASE.rstrip("/") + "/", url.lstrip("/"))
    ext = os.path.splitext(urlparse(url).path)[1].lower() or {"IMAGE": ".jpg", "VIDEO": ".mp4"}.get((m.get("type") or "").upper(), "")
    os.makedirs(MEDIA_DIR, exist_ok=True)
    path = f"{MEDIA_DIR}/post_{post_id}_media_{m.get('id')}{ext}"
    if not os.path.exists(path):
        # send the login token ONLY to our own backend, never to Cloudinary or other hosts
        headers = _headers() if urlparse(url).netloc == urlparse(API_BASE).netloc else {}
        with requests.get(url, headers=headers, stream=True, timeout=REQUEST_TIMEOUT) as r:
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(1 << 20): f.write(chunk)
    return path


In [16]:
# ===== Check every post of the user against the user's own skills =====
user, SKILLS, POSTS = normalize(fetch_user_data())
print(f"User {user.get('id')} ({user.get('name', '')}) | {len(SKILLS)} skills: {[s['name'] for s in SKILLS]} | {len(POSTS)} posts")

def check_post(post, skills, threshold=None):
    """One post (+ that user's skills) -> JSON-friendly dict.
    Each media item is checked on its own; the caption is checked too when it has real words (or when there is no media).
    Results are merged: a skill matches if any part matches, and its score is the best score. A failure becomes 'error'."""
    skill_id = {s["name"]: s["id"] for s in skills}
    parts, errors = [], []
    sources = [("media", m) for m in post["media"]]
    if len(post["text"].split()) >= 5 or not sources: sources.append(("text", post["text"]))
    for kind, item in sources:
        try:
            src = get_media(item, post["id"]) if kind == "media" else item
            if not src.strip(): raise ValueError("empty text")
            parts.append(check(src, skills, threshold))
        except Exception as e:
            errors.append(f"{kind}: {type(e).__name__}: {e}")
    if not parts:
        return {"post_id": post["id"], "error": "; ".join(errors) or "post has no text or media"}
    best = {}
    for r in parts:
        for row in r["table"].itertuples():
            best[row.skill] = max(best.get(row.skill, 0.0), float(row.judge))
    matches = sorted({n for r in parts for n in r["matches"]})
    out = {"post_id": post["id"],
           "topic": " | ".join(dict.fromkeys(r["label"] for r in parts)),
           "summary": " ".join(r["summary"] for r in parts),
           "related": any(r["related"] for r in parts),
           "matches": [{"skill_id": skill_id[n], "skill": n} for n in matches],
           "scores":  [{"skill_id": skill_id[n], "skill": n, "score": sc} for n, sc in best.items()]}
    if errors: out["warnings"] = errors
    return out

RESULTS = [check_post(p, SKILLS) for p in POSTS]

display(pd.DataFrame([{"post_id": r["post_id"], "topic": r.get("topic", ""), "related": r.get("related"),
                       "matched_skills": ", ".join(m["skill"] for m in r.get("matches", [])),
                       "error": r.get("error", "")} for r in RESULTS]))

payload = {"user_id": user.get("id"), "results": RESULTS}      # the shape you can later POST back to the backend
with open("/content/skill_check_results.json", "w") as f: json.dump(payload, f, indent=2)
print("Saved /content/skill_check_results.json")


Fetched from https://evolve-backend-4lux.onrender.com/api/skills/user/1/skills-posts
User 1 (a_mor) | 4 skills: ['Python', 'AI/ML', 'Web Development', 'Cloud'] | 1 posts


,post_id,topic,related,matched_skills,error
0,1,Login,True,Web Development,


Saved /content/skill_check_results.json


## Live mode: webhook

In [17]:
!pip install -q fastapi uvicorn requests

In [18]:
# ===== Webhook receiver: backend says "new post" -> we check it against that user's skills =====
import queue, threading, uvicorn
from fastapi import FastAPI, Header, HTTPException, Request

WEBHOOK_SECRET    = "change-me"     # same value in the backend's env; stops strangers calling the public URL
WEBHOOK_ALLOW_MOCK = False          # keep False: a real event must never be judged on fake data
SEND_RESULTS_BACK = False           # True once the backend team adds an endpoint that accepts the result
RESULT_ENDPOINT   = "/api/skills/user/{user_id}/posts/{post_id}/skill-match"   # PROPOSED: agree this with the backend team

JOBS = queue.Queue()                # one GPU = one worker, so jobs wait in line here
RESULTS_BY_POST = {}                # (user_id, post_id) -> result, kept in memory
app = FastAPI()

@app.get("/health")
def health():
    return {"ok": True, "queued": JOBS.qsize(), "done": len(RESULTS_BY_POST)}

@app.get("/results")
def results():
    return [{"user_id": u, "post_id": p, **r} for (u, p), r in RESULTS_BY_POST.items()]

@app.post("/webhook/new-post")
async def new_post(request: Request, x_webhook_secret: str = Header(default="")):
    if x_webhook_secret != WEBHOOK_SECRET: raise HTTPException(401, "bad secret")
    body = await request.json()
    uid = body.get("user_id")
    pid = body.get("post_id") or (body.get("post") or {}).get("id")
    if uid is None or pid is None: raise HTTPException(422, "need user_id and post_id")
    JOBS.put((uid, pid))
    return {"queued": True}          # answer fast; the slow model work happens in worker()

def process_job(uid, pid):
    user, skills, posts = normalize(fetch_user_data(uid, allow_mock=WEBHOOK_ALLOW_MOCK))
    post = next((p for p in posts if str(p["id"]) == str(pid)), None)
    if post is None: return {"post_id": pid, "error": "post not found in the API response"}
    return check_post(post, skills)

def send_back(uid, pid, res):
    url = API_BASE.rstrip("/") + RESULT_ENDPOINT.format(user_id=uid, post_id=pid)
    try: requests.post(url, json=res, headers=_headers(), timeout=REQUEST_TIMEOUT).raise_for_status()
    except Exception as e: print("send_back failed:", e)

def worker():
    while True:
        uid, pid = JOBS.get()
        try: res = process_job(uid, pid)
        except Exception as e: res = {"post_id": pid, "error": f"{type(e).__name__}: {e}"}
        RESULTS_BY_POST[(str(uid), str(pid))] = res
        print(f"[user {uid} | post {pid}] ->", res.get("matches", res.get("error")))
        if SEND_RESULTS_BACK and "error" not in res: send_back(uid, pid, res)
        JOBS.task_done()


In [19]:
# ===== Start the server + a public tunnel to it (run this cell once per Colab session) =====
import subprocess, re, os, time

PORT = 8000
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
threading.Thread(target=worker, daemon=True).start()

# cloudflared "quick tunnel": free, no account. (The ngrok free plan has only ONE dev domain, and you
# will use it for the backend on your laptop, so Colab gets cloudflared.)
if not os.path.exists("cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
                   " -O cloudflared && chmod +x cloudflared", shell=True, check=True)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
                          stderr=subprocess.PIPE, text=True)
PUBLIC_URL = None
for line in tunnel.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: PUBLIC_URL = m.group(0); break
threading.Thread(target=lambda: [None for _ in tunnel.stderr], daemon=True).start()   # keep the pipe drained
assert PUBLIC_URL, "cloudflared did not give a public URL"

print("Give this to the backend as the webhook URL:\n   ", PUBLIC_URL + "/webhook/new-post")
print("Header to send with it:  X-Webhook-Secret:", WEBHOOK_SECRET)
print("The URL changes every time this cell is re-run.")


Give this to the backend as the webhook URL:
    https://october-perspectives-highest-career.trycloudflare.com/webhook/new-post
Header to send with it:  X-Webhook-Secret: change-me
The URL changes every time this cell is re-run.


In [20]:
# ===== Test the webhook from inside Colab, with real data (user 1, first post). Needs a valid, unexpired token. =====
r = requests.post(f"http://localhost:{PORT}/webhook/new-post",
                  json={"user_id": USER_ID, "post_id": POSTS[0]["id"]},
                  headers={"X-Webhook-Secret": WEBHOOK_SECRET})
print(r.status_code, r.json())
for _ in range(30):                      # wait up to about a minute for the worker to finish
    time.sleep(2)
    if RESULTS_BY_POST: break
print(json.dumps(requests.get(f"http://localhost:{PORT}/results").json(), indent=2))


200 {'queued': True}
[]
